# Lab 03 — Call a model with the Bedrock Converse API

Original OfferReady lab. The **Converse API** gives one message shape across model
providers on Amazon Bedrock, so you swap models by changing an id. This notebook is
structured so it runs offline by default (mock) and hits real Bedrock when you flip
a flag and have AWS credentials.

**You will:** build a portable messages payload, call (or mock) Converse, and pull
the text out of the response.

## 1. Config

Never hardcode secrets. `boto3` picks up credentials from your environment / AWS
profile / role — not from code.

In [ ]:
import os, json

USE_REAL_BEDROCK = False           # flip to True with AWS creds configured
MODEL_ID = os.environ.get("MODEL_ID", "amazon.nova-lite-v1:0")
REGION = os.environ.get("AWS_REGION", "us-west-2")

## 2. Build a portable payload

Converse uses `messages` with typed content blocks and an `inferenceConfig`
(temperature, maxTokens). The same shape works across providers.

In [ ]:
def build_request(user_text, system_text=None, temperature=0.2, max_tokens=300):
    req = {
        "modelId": MODEL_ID,
        "messages": [{"role": "user", "content": [{"text": user_text}]}],
        "inferenceConfig": {"temperature": temperature, "maxTokens": max_tokens},
    }
    if system_text:
        req["system"] = [{"text": system_text}]
    return req

req = build_request(
    "Explain retrieval-augmented generation in two sentences.",
    system_text="You are a concise technical assistant.",
)
print(json.dumps(req, indent=2))

## 3. Call (or mock) Converse

The response shape is `output.message.content[0].text`.

In [ ]:
def converse(req):
    if not USE_REAL_BEDROCK:
        # Offline mock so the lab runs with no AWS account.
        return {"output": {"message": {"content": [
            {"text": "RAG retrieves relevant text from your data and gives it to the"
                     " model as context. The model then answers from that context,"
                     " reducing hallucination and enabling citations."}
        ]}}}
    import boto3
    brt = boto3.client("bedrock-runtime", region_name=REGION)
    return brt.converse(**req)

def extract_text(resp):
    return resp["output"]["message"]["content"][0]["text"]

resp = converse(req)
print(extract_text(resp))

## 4. Wire it into RAG

Replace the `fake_llm` in Lab 01 with a Converse call: build the grounded prompt,
pass it as the user message, extract the text. Same pipeline, real model.

In [ ]:
def answer_grounded(question, context):
    system = "Answer ONLY from the provided context. Cite sources. If unknown, say so."
    user = f"<context>\n{context}\n</context>\n\nQuestion: {question}"
    return extract_text(converse(build_request(user, system_text=system)))

print(answer_grounded(
    "How long do refunds take?",
    "[refunds] Refunds are processed within 5 business days.",
))

## Notes

- Enable the model in your Bedrock account/region before real calls.
- Use **temperature 0** for extraction/JSON; give enough `maxTokens` to avoid truncation.
- Add **Guardrails** for input/output safety; wrap calls with timeout + retry (Study Guide Ch3).

See the Study Guide, Chapter 10 (Bedrock & AgentCore).